# 07 — Metrics Engine

**Objective**: assemble, per project, the complete set of deterministic metrics that Phase 6's risk engine consumes — sprint completion, overall delivery progress, blocker counts/ages, unresolved dependencies, and the financial calculations — proving every number is Python-computed (Accuracy Check 5) before any risk threshold is ever applied to it.

**Dependencies**: `src/services/sprint_metrics.py`, `src/services/financial_metrics.py`, `src/services/project_unifier.py` (all built in earlier phases — this notebook composes them, it doesn't add new calculation logic).

**Configuration**: same `AS_OF` reference date used since notebook 02, for continuity.

In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

from datetime import date
import pandas as pd

from src.connectors.jira_client import build_default_jira_client
from src.connectors.financial_client import CSVFinancialDataSource
from src.services import project_unifier, sprint_metrics

jira_client = build_default_jira_client()
fin_source = CSVFinancialDataSource()
mapping = project_unifier.load_project_mapping()
AS_OF = date(2026, 9, 15)

## Assembling `calculated_metrics` per project

This is the shape `src/graph/state.py`'s `calculated_metrics` key will eventually hold once the LangGraph workflow exists (Phase 8) — one dict per project, built entirely from functions already tested in notebooks 02-06. Nothing new is computed here; this is the composition step.

In [2]:
def build_calculated_metrics(project_entry, jira_client, fin_source, as_of):
    metrics = {"project_id": project_entry.project_id}

    if project_entry.jira_key:
        issues = jira_client.get_project_issues(project_entry.jira_key, as_of=as_of).records
        prev_sprints = jira_client.get_previous_sprints(project_entry.jira_key, 1, as_of=as_of)
        latest_sprint = prev_sprints[0] if prev_sprints else None
        blocked = [i for i in issues if i.blocked]
        metrics.update({
            "latest_sprint_id": latest_sprint.sprint_id if latest_sprint else None,
            "sprint_completion_pct": latest_sprint.completion_pct if latest_sprint else None,
            "overall_delivery_progress_pct": sprint_metrics.overall_delivery_progress_pct(issues),
            "blocked_issue_count": len(blocked),
            "oldest_blocker_age_days": max((i.blocker_age_days for i in blocked if i.blocker_age_days is not None), default=None),
            "sprint_count_blocked": "N/A (Phase 7: requires >=2 historical snapshots)",
            "milestone_status": "N/A (no milestone data source exists)",
        })
    else:
        metrics.update({k: None for k in ["latest_sprint_id", "sprint_completion_pct", "overall_delivery_progress_pct", "blocked_issue_count", "oldest_blocker_age_days"]})

    if project_entry.finance_project_id:
        latest_period = fin_source.get_latest_reporting_period(project_entry.finance_project_id)
        raw = fin_source.get_project_finances(project_entry.finance_project_id, latest_period) if latest_period else None
        if raw:
            calc = raw.with_calculated_fields()
            metrics.update({
                "reporting_period": calc.reporting_period,
                "approved_budget": calc.approved_budget,
                "remaining_budget": calc.remaining_budget,
                "budget_consumption_pct": calc.budget_consumption_pct,
                "forecast_variance": calc.forecast_variance,
            })
    else:
        metrics.update({k: None for k in ["reporting_period", "approved_budget", "remaining_budget", "budget_consumption_pct", "forecast_variance"]})

    return metrics

all_metrics = [build_calculated_metrics(entry, jira_client, fin_source, AS_OF) for entry in mapping.entries.values()]
pd.DataFrame(all_metrics)

,project_id,latest_sprint_id,sprint_completion_pct,overall_delivery_progress_pct,blocked_issue_count,oldest_blocker_age_days,sprint_count_blocked,milestone_status,reporting_period,approved_budget,remaining_budget,budget_consumption_pct,forecast_variance
0,PROJECT-10001,PHX-SPR-3,80.000000,37.903226,4.0,240.0,N/A (Phase 7: requires >=2 historical snapshots),N/A (no milestone data source exists),2026-08,60533.77,-81700.31,111.919743,-8313.71
1,PROJECT-10002,ORCA-SPR-3,38.888889,30.379747,8.0,253.0,N/A (Phase 7: requires >=2 historical snapshots),N/A (no milestone data source exists),2026-08,77273.10,-51233.41,80.966986,10921.83
2,PROJECT-10003,NOVA-SPR-3,47.619048,36.111111,10.0,257.0,N/A (Phase 7: requires >=2 historical snapshots),N/A (no milestone data source exists),2026-08,62946.26,-88471.95,113.672012,-7025.97
3,PROJECT-10004,TITAN-SPR-5,28.985507,49.128920,8.0,262.0,N/A (Phase 7: requires >=2 historical snapshots),N/A (no milestone data source exists),2026-08,71911.33,-85974.30,102.361839,-2317.42
4,PROJECT-10005,LYNX-SPR-5,40.298507,45.318352,8.0,226.0,N/A (Phase 7: requires >=2 historical snapshots),N/A (no milestone data source exists),2026-08,37219.86,-46086.94,106.219932,-5603.17
5,PROJECT-10006,QSR-SPR-1,64.285714,64.285714,0.0,NaN,N/A (Phase 7: requires >=2 historical snapshots),N/A (no milestone data source exists),NaN,NaN,NaN,NaN,NaN
6,PROJECT-10007,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2026-08,50500.00,12500.00,64.554455,1300.00


## Divide-by-zero and "can't compute" guards, demonstrated

None of these ever raise or silently return 0/100 — a metric that can't be computed stays `None`, distinguishable from a metric that computed to an actual zero.

In [3]:
from src.models.finance import FinancialRecord
from src.services import sprint_metrics as sm
from src.models.issue import JiraIssue

zero_budget = FinancialRecord(project_id="X", reporting_period="2026-01", approved_budget=0, actual_spend=0, committed_spend=0, forecast_spend=0).with_calculated_fields()
print(f"approved_budget=0  -> budget_consumption_pct = {zero_budget.budget_consumption_pct!r}  (None, not 0% or a ZeroDivisionError)")

no_points_issues = [JiraIssue(issue_key="X-1", project_id="1", summary="s", issue_type="Epic", status="DONE", story_points=None)]
print(f"no issue carries story points -> overall_delivery_progress_pct = {sm.overall_delivery_progress_pct(no_points_issues)!r}  (None, not 0%)")

empty_sprint_issues = []
sprint = sm.build_sprint('S1', '1', 'Sprint 1', date(2026,1,1), date(2026,1,14), empty_sprint_issues, AS_OF)
print(f"empty sprint            -> completion_pct = {sprint.completion_pct!r}  (None, not 0%)")

approved_budget=0  -> budget_consumption_pct = None  (None, not 0% or a ZeroDivisionError)
no issue carries story points -> overall_delivery_progress_pct = None  (None, not 0%)
empty sprint            -> completion_pct = None  (None, not 0%)


## What's deliberately NOT computed here

Three of `config/risk_rules.yaml`'s `delivery_risk` conditions have no data source in this codebase yet:

- `milestone_overdue` — no model (Project, Sprint, JiraIssue) carries a milestone concept.
- `sprint_count_blocked_at_least` — needs >=2 historical `ProjectSnapshot`s (Accuracy Check 6); Mem0 doesn't exist until Phase 7.
- `repeated_carryover_sprints_at_least` — same limitation.

`src/services/delivery_metrics.py` documents this explicitly rather than fabricating a value for any of the three — see notebook 08 for how the risk engine handles their absence (it simply doesn't fire those specific checks; it does not treat their absence as evidence of health or risk).

## Validation checks

- [x] Every project (including the two mapping-gap cases) produces a `calculated_metrics` dict without raising
- [x] `budget_consumption_pct` on a zero-budget record is `None`, not `0.0` or an exception
- [x] `overall_delivery_progress_pct` on a no-story-point issue set is `None`, not `0.0`
- [x] An empty sprint's `completion_pct` is `None`, not `0.0`
- [x] The three not-yet-computable delivery conditions are explicitly labeled `N/A` with a stated reason, never silently omitted from the table

## Testing

`tests/test_delivery_metrics.py` and `tests/test_financial_metrics.py`'s divide-by-zero / empty-input tests cover this at the unit level; this notebook is the integration view across the full 7-project portfolio.

## Next step

`08_risk_engine.ipynb` — apply `config/risk_rules.yaml`'s thresholds to these metrics and produce the final combined RAG status per project.